[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Connecting to a Hosted Server &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup. It generates the certificate and turns the server's TLS on,
so run it first. Each task then stands on its own and they can be run in any order.


In [1]:
import asyncio
import getpass
import os
import pathlib
import secrets
import ssl
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import conninfo, sql

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

TLS = pathlib.Path("/tmp/guide_tls")                       # one place, on any machine
CERTIFICATE, KEY = TLS / "server.crt", TLS / "server.key"
RULE = "hostssl guide  app  127.0.0.1/32  scram-sha-256"


def as_server_owner():
    """The prefix that lets this notebook edit the server's own files, where one is needed."""
    return "" if sys.platform != "linux" or os.geteuid() == 0 else "sudo "


def enable_tls(wait=30):
    """Generate a self-signed certificate and switch the server's TLS on. Idempotent."""
    TLS.mkdir(exist_ok=True)
    TLS.chmod(0o755)
    if not CERTIFICATE.exists():
        shell(f'openssl req -new -x509 -days 365 -nodes -out "{CERTIFICATE}" -keyout "{KEY}" '
              f'-subj "/CN=127.0.0.1" -addext "subjectAltName=IP:127.0.0.1"')
    KEY.chmod(0o600)                                          # PostgreSQL refuses a readable key
    if sys.platform == "linux":
        shell(f'{as_server_owner()}chown postgres "{KEY}" "{CERTIFICATE}"')

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute(sql.SQL("ALTER SYSTEM SET ssl_cert_file = {}")
                     .format(sql.Literal(str(CERTIFICATE))))
        conn.execute(sql.SQL("ALTER SYSTEM SET ssl_key_file = {}").format(sql.Literal(str(KEY))))
        conn.execute("ALTER SYSTEM SET ssl = on")
        conn.execute("SELECT pg_reload_conf()")               # ssl is sighup, so no restart

    for attempt in range(wait):
        with psycopg.connect("dbname=guide") as conn:
            if conn.execute("SHOW ssl").fetchone()[0] == "on":
                return "on"
        time.sleep(1)
    raise RuntimeError("The server did not switch TLS on. Check its log for the certificate.")


def hba_rule(present):
    """Add or remove one line of pg_hba.conf, for one role, on one address."""
    with psycopg.connect("dbname=guide") as conn:
        path = conn.execute("SHOW hba_file").fetchone()[0]
    edit = (f"p.write_text('{RULE}\\n' + p.read_text())" if present else
            f"p.write_text(''.join(l for l in p.read_text().splitlines(True) "
            f"if l.strip() != '{RULE}'))")
    shell(f"""{as_server_owner()}python3 -c "
import pathlib
p = pathlib.Path('{path}')
if ('{RULE}' in p.read_text()) != {present}:
    {edit}
" """)
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("SELECT pg_reload_conf()")


def connected(conninfo_string, **kwargs):
    """Try one connection and say in one line what happened, without a traceback."""
    try:
        with psycopg.connect(conninfo_string, **kwargs) as conn:
            encrypted = conn.execute("SELECT ssl FROM pg_stat_ssl "
                                     "WHERE pid = pg_backend_pid()").fetchone()[0]
            return f"connected, encrypted={encrypted}"
    except psycopg.OperationalError as error:
        return str(error).splitlines()[0].split("failed: ")[-1]


print("server:", start_server())
print(report())
print("TLS:", enable_tls())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
TLS: on


**1.** A URL, apart and back together.


In [2]:
url = "postgresql://someone@db.example.com:5432/production?sslmode=verify-full"

settings = conninfo.conninfo_to_dict(url)
print("apart:  ", settings)
print("together:", conninfo.make_conninfo("", **settings))


apart:   {'user': 'someone', 'dbname': 'production', 'host': 'db.example.com', 'port': '5432', 'sslmode': 'verify-full'}
together: user=someone dbname=production host=db.example.com port=5432 sslmode=verify-full


The rebuilt string is the keyword spelling of the same connection. Going through the dictionary is
how you change one part of a DSN without string surgery on a URL.


**2.** Encrypted, and asked rather than assumed.


In [3]:
with psycopg.connect("host=127.0.0.1 dbname=guide sslmode=require") as conn:
    print(conn.execute("SELECT ssl, version, cipher FROM pg_stat_ssl "
                       "WHERE pid = pg_backend_pid()").fetchone())


(True, 'TLSv1.3', 'TLS_AES_256_GCM_SHA384')


`pg_stat_ssl` is the server's own view of the connection, which is worth more than the client's
settings: it is the answer to "is this actually encrypted" rather than "did I ask for encryption".


**3.** Verified, against the certificate Setup made.


In [4]:
verified = f"host=127.0.0.1 dbname=guide sslmode=verify-full sslrootcert={CERTIFICATE}"

with psycopg.connect(verified) as conn:
    print("verify-full connected:", conn.execute("SELECT ssl FROM pg_stat_ssl "
                                                 "WHERE pid = pg_backend_pid()").fetchone())
print("by name instead of address:", connected(verified.replace("127.0.0.1", "localhost")))


verify-full connected: (True,)
by name instead of address: server certificate for "127.0.0.1" (and 1 other name) does not match host name "localhost"


The second line is the check `verify-full` adds. The certificate is made out to the address, so the
name does not match it, and that is exactly the check that catches a server pretending to be yours.


**4.** The same, in asyncpg.


In [5]:
context = ssl.create_default_context(cafile=str(CERTIFICATE))

conn = await asyncpg.connect(host="127.0.0.1", database="guide", ssl=context)
print("asyncpg, verified:", await conn.fetchval(
    "SELECT ssl FROM pg_stat_ssl WHERE pid = pg_backend_pid()"))
await conn.close()


asyncpg, verified: True


A default context verifies the chain and the hostname, so passing one with the right `cafile` is
asyncpg's `verify-full`. There is no mode name to get wrong.


**5.** Nothing but the environment.


In [6]:
os.environ.update({"PGHOST": "127.0.0.1", "PGDATABASE": "guide", "PGSSLMODE": "require"})

with psycopg.connect() as conn:
    print("connected to", conn.execute("SELECT current_database()").fetchone(),
          "encrypted:", conn.execute("SELECT ssl FROM pg_stat_ssl "
                                     "WHERE pid = pg_backend_pid()").fetchone()[0])

for name in ("PGHOST", "PGDATABASE", "PGSSLMODE"):
    del os.environ[name]


connected to ('guide',) encrypted: True


libpq reads those on its own, so a program can be written with no connection details in it at all
and configured entirely from outside. `PGPASSWORD` works the same way and is where a password goes
when `~/.pgpass` is not available.


**6.** Two seconds, then give up.


In [7]:
UNROUTABLE = "10.255.255.1"

start = time.perf_counter()
print("psycopg:", connected(f"host={UNROUTABLE} dbname=guide connect_timeout=2"),
      f"after {time.perf_counter() - start:.0f}s")

start = time.perf_counter()
try:
    await asyncpg.connect(host=UNROUTABLE, database="guide", timeout=2)
except TimeoutError:
    print("asyncpg: TimeoutError with no message,",
          f"after {time.perf_counter() - start:.0f}s")


psycopg: connection timeout expired after 2s
asyncpg: TimeoutError with no message, after 2s


Two seconds each, and without those arguments both would have waited for the operating system to
give up, which can be minutes. Put one in every DSN that leaves the machine.


---

&#8592; **Back to:** [Connecting to a Hosted Server](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/14-connecting-to-a-hosted-server.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
